# Chapter 11: Reinforcement Learning

This notebook accompanies **Chapter 11** of the lecture notes.

> Last lecture the supervision was a label per input: partial, noisy, conflicting, but always pointing at "the right answer". This time the supervision is a single scalar reward, often delayed, often zero, and the agent has to figure out by itself which of its many actions earned it. The chapter formalises the setting as a Markov decision process, derives the optimal solution when we know the dynamics (value iteration), then drops that knowledge and recovers the same policy from interaction alone (Q-learning). Around the Q-learning loop we turn the knobs that decide whether real RL succeeds or stalls: how much to explore, how dense the reward needs to be, whether to reuse past experience. We do all of this on a tiny Pong-like catcher small enough that **value iteration finds the optimal policy in milliseconds**, giving us a ground truth to measure every subsequent method against.

**Agenda**

🗺️ · 🎯 · 🏁

**Take it from here:** 📈 · 🏟️

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones. Everything here is pure NumPy; the entire notebook runs in well under a minute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '../..')
from plot_style import *
from checks import (
    check_value_iteration, check_q_learning_update, check_policy_gradient, check_pick_epsilon_greedy,
)
from viz_helpers import (
    MiniPongEnv, W, H, N_ACTIONS,
    N_STATES, ACTION_NAMES, state_index, build_mdp,
    features, policy_probs, bellman_backup, td_target,
    compute_returns, evaluate_policy, greedy_action_from_Q, train_q_learning,
    train_reinforce, plot_value_slice, plot_learning_curve, plot_gradient_norms,
    plot_visitation, collect_visitation, epsilon_greedy_using, expected_value_under,
    plot_env_schematic, plot_baseline_advantage_trace, play_live,
)

RNG = np.random.default_rng(0)

## 🗺️ Setting up mini-Pong

Mini-Pong is a tiny Markov decision process: state `(bx, by, vy, py)` for ball position, ball vertical velocity, and paddle row; three actions (up, stay, down); reward only at the end of each rally (+1 catch, -1 miss). The state space is small enough that we can compute the optimal value function `V*` exactly with value iteration. That gives us an *upper bound* to measure every learned agent against. After this section we throw `(P, R)` away and learn from interaction alone.

In [ ]:
env = MiniPongEnv()
print(f'grid                 : {W} columns × {H} rows  (paddle column = {W - 1})')
print(f'state space size     : {N_STATES} (= {(W - 1) * H * 2 * H} non-terminal + 1 terminal)')
print(f'action space         : {N_ACTIONS}  → {ACTION_NAMES}')
print(f'discount factor γ    : 0.95   (sparse terminal reward; γ controls patience)')
print()
s = env.reset(seed=0)
print(f'an example reset state: (bx, by, vy, py) = {s}')


In [ ]:
plot_env_schematic()


### What random play looks like

Before training anything, watch a uniformly random policy. The paddle wanders, the ball lands wherever it lands, and the rally counter at the top almost never gets past one. This is the floor we will measure improvement against.

In [ ]:
rng_random = np.random.default_rng(0)
await play_live(
    MiniPongEnv(),
    lambda s: int(rng_random.integers(0, N_ACTIONS)),
    fps=4, max_rallies=20, seed=42,
    title='random policy: live play')

### Value iteration: an exact optimum to aim at

When we know the transition tensor `P` and reward tensor `R`, the Bellman equation lets us compute the optimal value function directly. Iterate the Bellman backup, take the max over actions, repeat. This converges to `V*` for any discounted MDP. We never get this luxury in a real environment, but on mini-Pong it gives us ground truth.

In [ ]:
def value_iteration(P, R, gamma, n_iters):
    """Iterate Bellman backups; take max over actions to update V."""
    V = np.zeros(P.shape[0])
    for _ in range(n_iters):
        Q = bellman_backup(V, P, R, gamma)
        V = Q.max(axis=1)
    return V


check_value_iteration(value_iteration)


In [ ]:
# Build the MDP tensors and run value iteration.
P, R = build_mdp()
print(f'P shape : {P.shape}     (transitions; each P[s, a, :] is a one-hot)')
print(f'R shape : {R.shape}     (rewards)')

V_star = value_iteration(P, R, gamma=0.95, n_iters=200)
Q_star = bellman_backup(V_star, P, R, gamma=0.95)

print()
print(f'V*  range : [{V_star.min():.3f}, {V_star.max():.3f}]')
print(f'V*  averaged over the initial-state distribution : '
      f'{expected_value_under(V_star):.3f}')
print(f'  (this is the optimal expected return: the ceiling every method below is chasing.)')


In [ ]:
# Greedy policy from Q*. Sanity-check the score over 500 fresh episodes.
def optimal_action(state):
    return int(np.argmax(Q_star[state_index(state)]))

print(f'optimal policy averaged over 500 random episodes: '
      f'{evaluate_policy(optimal_action, n_episodes=500, seed=0):+.3f}  (expected near +1.0)')

In [ ]:
plot_value_slice(V_star, title='V*  (slice: paddle centred, vy = +1)')


### What optimal play looks like

The greedy policy extracted from `Q*` is the patient-best paddle: under any starting ball, it has time to reach the impact column. Watching it play is the *upper bound* the rest of the chapter is trying to recover from interaction alone.

In [ ]:
await play_live(
    MiniPongEnv(), optimal_action,
    fps=4, max_rallies=20, seed=42,
    title='optimal policy (greedy w.r.t. Q*): live play')

## 🎯 Q-learning

Value iteration solves the MDP exactly when we *know* the transition tensor `P` and reward tensor `R`. In every real environment we do not. Q-learning estimates the optimal action-value `Q*(s, a)` from interaction alone: each transition `(s, a, r, s_next)` nudges `Q[s, a]` toward `r + γ · max_a Q[s_next, a]`. The same Bellman idea, now sample-driven.

The next four experiments run this loop and turn knobs students need to *feel*: exploration strength, reward density, and the use of an experience replay buffer. After each one the trained agent plays a live game against the env so the score has a face attached.

### Q-learning update

`td_target(r, gamma, q_next_max, done)` is provided as a black-box helper: it computes `r + γ · max_a' Q(s', a')`, except at terminal transitions where the bootstrap term is dropped. Implement `q_learning_update(Q, s, a, r, s_next, done, alpha, gamma)`: one off-policy TD update on a single transition. The rule is:
```
Q[s, a] ← Q[s, a] + α · (target − Q[s, a])
```
where `target = td_target(r, gamma, max_{a'} Q[s_next, a'], done)`. Mutate `Q` in place (or return a new array; the auto-checker accepts either).

In [ ]:
def q_learning_update(Q, s, a, r, s_next, done, alpha, gamma):
    """One off-policy TD update on a single transition.  Returns the updated Q."""
    q_next_max = Q[s_next].max()
    target = td_target(r, gamma, q_next_max, done)
    Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])
    return Q


check_q_learning_update(q_learning_update)


### Experiment A: train one agent and watch it play

Hyperparameters first: discount `γ = 0.95` (patient enough for a 6-step rally), learning rate `α = 0.10` (small enough not to overshoot Q on a single transition), and exploration `ε = 0.10` (a 1-in-10 chance the agent picks a random action while training). Vanilla Q-learning consumes each transition once, with no replay.

In [ ]:
GAMMA = 0.95
ALPHA = 0.10
EPS   = 0.10

Q_vanilla, returns_vanilla = train_q_learning(
    q_learning_update, alpha=ALPHA, gamma=GAMMA, eps=EPS,
    n_episodes=2000, use_replay=False, seed=0,
)

plot_learning_curve({"vanilla Q-learning": returns_vanilla},
                    title='Q-learning: episode return per training episode',
                    smooth_k=50)

score_vanilla = evaluate_policy(greedy_action_from_Q(Q_vanilla),
                                 n_episodes=500, seed=0)
print(f'greedy(Q_vanilla) over 500 fresh episodes: {score_vanilla:+.3f}')

Now watch it play. Each frame is rendered live as the cell executes; the rally counter at the top tracks consecutive catches. A score around +1.00 on 500 episodes should translate into a long live streak; a lower score will miss in an earlier rally.

In [ ]:
await play_live(MiniPongEnv(), greedy_action_from_Q(Q_vanilla),
                fps=4, max_rallies=20, seed=42,
                title='vanilla Q-learning: live play')

### Experiment B: exploration vs exploitation

ε-greedy picks the greedy action with probability `1 - ε` and a uniformly random action with probability `ε`. The trade-off is sharp: ε too low and the agent commits early to whatever happened to work in its first episodes, never trying the alternative that actually generalises; ε too high and every step is half noise, so even after training the greedy policy you extract is shaped by stale arbitrary updates. Below we train three Q-tables under three different ε on identical hyperparameters and look at *what each one explored*, *what each one learned*, and *how each one scores*.

In [ ]:
def pick_epsilon_greedy(Q_row, eps, rng):
    """ε-greedy action: random with prob eps, argmax otherwise."""
    if rng.random() < eps:
        return int(rng.integers(0, len(Q_row)))
    return int(np.argmax(Q_row))


check_pick_epsilon_greedy(pick_epsilon_greedy)


In [ ]:
# Three exploration strengths, identical otherwise.
kw = dict(alpha=ALPHA, gamma=GAMMA, n_episodes=2000, seed=0)
Q_eps00, ret_eps00 = train_q_learning(q_learning_update, eps=0.00, **kw)
Q_eps10, ret_eps10 = train_q_learning(q_learning_update, eps=0.10, **kw)
Q_eps40, ret_eps40 = train_q_learning(q_learning_update, eps=0.40, **kw)

plot_learning_curve({
    "ε = 0.00 (pure greedy)" : ret_eps00,
    "ε = 0.10"               : ret_eps10,
    "ε = 0.40 (aggressive)"  : ret_eps40,
}, title='ε sweep: episode return per training episode', smooth_k=50)

for label, Q in [("ε=0.00", Q_eps00), ("ε=0.10", Q_eps10), ("ε=0.40", Q_eps40)]:
    s = evaluate_policy(greedy_action_from_Q(Q), n_episodes=500, seed=0)
    print(f'greedy({label}): {s:+.3f}')

### What each ε actually visited

A learning curve says how much reward each rule earned during training. The visitation heatmaps say *where on the grid* it spent its episodes. Pure greedy hugs a thin diagonal because it never tried anything else; ε = 0.4 spreads visits across the whole grid but never commits long enough to a successful policy. The middle ε is the one that turns exploration into a learned policy.

In [ ]:
# Visitation under each behaviour rule on the default (dense) reward env.
def visits_under(Q, eps):
    rng = np.random.default_rng(0)
    return collect_visitation(
        epsilon_greedy_using(pick_epsilon_greedy, Q, eps, rng),
        n_episodes=300, seed=0)[0]

plot_visitation(visits_under(Q_eps00, 0.00),
                title='visitation, ε = 0.00  (pure greedy)')
plot_visitation(visits_under(Q_eps10, 0.10),
                title='visitation, ε = 0.10')
plot_visitation(visits_under(Q_eps40, 0.40),
                title='visitation, ε = 0.40  (aggressive)')

**Observe:** the U-shape across ε is the central lesson. ε = 0 never explored, so its greedy extraction is whatever the first few seeds rewarded; ε = 0.4 explored thoroughly but never committed long enough for the value function to consolidate. The middle ε wins both: enough exploration to find the right transitions, enough exploitation to reinforce them.

### Experiment C: reward shaping (sparse vs dense)

Default mini-Pong gives `-1` on miss and `+1` on hit, so every miss is a learning moment. The *sparse* variant gives `0` on miss instead, leaving only catches as signal. Real environments live closer to the sparse end (game scores tick rarely; molecules either bind or do not). The same algorithm on the same env, with the only difference being miss reward, looks very different.

In [ ]:
# Same algorithm, two reward variants. Sparse env: misses give 0 instead of -1.
kw = dict(alpha=ALPHA, gamma=GAMMA, eps=EPS, n_episodes=2000, seed=0)

Q_dense,  ret_dense  = train_q_learning(q_learning_update, sparse=False, **kw)
Q_sparse, ret_sparse = train_q_learning(q_learning_update, sparse=True,  **kw)

plot_learning_curve({
    "dense reward (-1 on miss)": ret_dense,
    "sparse reward (0 on miss)": ret_sparse,
}, title='Reward shaping: episode return per training episode', smooth_k=50)

score_dense  = evaluate_policy(greedy_action_from_Q(Q_dense),
                                n_episodes=500, seed=0)
score_sparse = evaluate_policy(greedy_action_from_Q(Q_sparse),
                                n_episodes=500, seed=0, sparse=True)
print(f'greedy(dense)  on dense env  : {score_dense:+.3f}')
print(f'greedy(sparse) on sparse env : {score_sparse:+.3f}')

**Observe:** dense climbs faster because every miss carries usable signal. Sparse stays flat until the agent stumbles onto its first catch, after which the value function starts to spread. Reward shaping is "give the agent more frequent gradient information"; in production that often means engineering a shaping term, with the cost that bad shaping rewards the wrong thing.

### Experiment D: experience replay

Vanilla Q-learning consumes each transition once. *Replay* stores recent transitions in a buffer and samples small batches from it on each step, so each transition gets reused and consecutive updates de-correlate. On this small MDP the gap is modest. On Atari, this single change is what made tabular ideas scalable to deep Q-networks.

In [ ]:
kw = dict(alpha=ALPHA, gamma=GAMMA, eps=EPS, n_episodes=2000, seed=0)

Q_plain,  returns_plain  = train_q_learning(q_learning_update,
                                              use_replay=False, **kw)
Q_replay, returns_replay = train_q_learning(q_learning_update,
                                              use_replay=True,
                                              buffer_size=2000,
                                              batch_size=8, **kw)

plot_learning_curve({
    "no replay"   : returns_plain,
    "with replay" : returns_replay,
}, title='Replay vs no-replay: episode return per training episode', smooth_k=50)

print(f'greedy(no replay)   : '
      f'{evaluate_policy(greedy_action_from_Q(Q_plain),  n_episodes=500, seed=0):+.3f}')
print(f'greedy(with replay) : '
      f'{evaluate_policy(greedy_action_from_Q(Q_replay), n_episodes=500, seed=0):+.3f}')
print(f'optimal (V*)        : '
      f'{evaluate_policy(optimal_action,                 n_episodes=500, seed=0):+.3f}')

### 🪜 Tune and watch: chase a perfect game

You have four knobs: learning rate, exploration ε, training-episode budget, and the replay flag. Pick a target (twenty rallies cleared on multiple seeds), edit the values below, and watch what the failure mode actually looks like. The number after `evaluate_policy` is the aggregate score; the live game is the per-episode story.

In [ ]:
# Edit and re-run. Tuning suggestions: alpha in [0.05, 0.40], eps in [0.05, 0.30],
# n_episodes in [500, 5000], replay on or off.
ALPHA_TRY  = 0.10
EPS_TRY    = 0.10
N_EPS_TRY  = 2000
REPLAY_TRY = True

Q_try, _ = train_q_learning(
    q_learning_update,
    alpha=ALPHA_TRY, gamma=GAMMA, eps=EPS_TRY,
    n_episodes=N_EPS_TRY, use_replay=REPLAY_TRY,
    buffer_size=2000, batch_size=8, seed=0,
)

score_try = evaluate_policy(greedy_action_from_Q(Q_try),
                             n_episodes=500, seed=0)
print(f'evaluate_policy (500 episodes): {score_try:+.3f}')

In [ ]:
await play_live(MiniPongEnv(), greedy_action_from_Q(Q_try),
                fps=4, max_rallies=20, seed=42,
                title='your tuned Q-learning agent: live play')

### 🏁 Recap

- 🗺️ The MDP is `(S, A, P, R, γ)`; given `P` and `R`, value iteration recovers the optimal value function and policy by repeatedly applying the Bellman backup.
- 🎯 Q-learning recovers the same `Q*` from interaction alone, by bootstrapping each transition toward `r + γ · max Q[s', ·]`.
- The four experiments above ran the same Q-learning loop while varying one knob at a time: exploration strength, reward density, and replay. Each is a lever that real DQN tunes carefully; the toy env makes the gradients of each lever visible in seconds.

The next chapter steps from a known finite MDP into one where the state space is too big to enumerate, and the Q-table becomes a function approximator.

## Take It from Here, Next Steps

Two parallel tracks. Pick one based on where you want to dig deeper. The classical track stays inside the planning view and adds a second fixed-point scheme; the modern track picks up the engineering knobs (trust regions, exploration, function approximation) that get RL to scale.

### 📈 Policy gradient: REINFORCE

Q-learning learns a value function and *extracts* a policy by taking argmax. REINFORCE goes the other way: it parameterises the policy directly and updates it to make rewarded actions more likely. Two consequences. The policy can stay stochastic, which matters when the optimal play involves randomness (poker) or when a continuous action space rules out argmax. The gradient is high variance, which a baseline tames by subtracting a running average return so the update reflects how much *better than expected* this episode was, rather than its absolute size.

In [ ]:
def policy_gradient(theta, phis, actions, advantages):
    """Sum of advantage-weighted ∇log π over a trajectory."""
    n_actions, n_features = theta.shape
    grad = np.zeros_like(theta)
    for phi, a, adv in zip(phis, actions, advantages):
        probs = policy_probs(theta, phi)
        indicator = np.zeros(n_actions); indicator[int(a)] = 1.0
        grad += float(adv) * np.outer(indicator - probs, phi)
    return grad


check_policy_gradient(policy_gradient)


In [ ]:
theta_pg, returns_pg, grad_norms_pg = train_reinforce(
    compute_returns, policy_gradient,
    lr=0.10, gamma=GAMMA, n_episodes=1200, baseline=False, seed=0,
)
theta_pg_b, returns_pg_b, grad_norms_pg_b = train_reinforce(
    compute_returns, policy_gradient,
    lr=0.10, gamma=GAMMA, n_episodes=1200, baseline=True, seed=0,
)

def policy_action(theta):
    return lambda s: int(np.argmax(theta @ features(s)))

print('Average return on 500 fresh episodes under each policy:')
print(f'  REINFORCE                 : '
      f'{evaluate_policy(policy_action(theta_pg), n_episodes=500, seed=0):+.3f}')
print(f'  REINFORCE + baseline      : '
      f'{evaluate_policy(policy_action(theta_pg_b), n_episodes=500, seed=0):+.3f}')
print(f'  optimal                   : '
      f'{evaluate_policy(optimal_action, n_episodes=500, seed=0):+.3f}')


In [ ]:
plot_learning_curve({
    'REINFORCE'              : returns_pg,
    'REINFORCE + baseline'   : returns_pg_b,
}, title='Policy gradient: episode return', smooth_k=30)

plot_gradient_norms({
    'no baseline'      : grad_norms_pg,
    'with baseline'    : grad_norms_pg_b,
}, title='Gradient-norm trajectories: baseline reduces variance')


### Baseline turns *return* into *advantage*

The baseline subtracts a running average of recent returns. Top panel shows the no-baseline run's return curve next to the slow-moving baseline `b̄_t` itself; bottom panel shows the per-episode advantage each run feeds into the gradient. The baselined advantage hovers around zero with a much smaller amplitude; that's the variance reduction the chapter promised.

In [ ]:
plot_baseline_advantage_trace(returns_pg, returns_pg_b, gamma=GAMMA)


### Watching a stochastic policy

REINFORCE produces a softmax over actions, not an argmax. The agent's *play* is therefore stochastic: at every step it samples an action from `policy_probs(theta, features(s))`. Watching that for twenty rallies makes the variance visible. The same state will sometimes be played differently, which is the whole reason policy gradient methods need a baseline.

In [ ]:
def policy_action_stochastic(theta, seed=0):
    rng = np.random.default_rng(seed)
    def f(s):
        p = policy_probs(theta, features(s))
        return int(rng.choice(N_ACTIONS, p=p))
    return f

await play_live(
    MiniPongEnv(),
    policy_action_stochastic(theta_pg_b, seed=0),
    fps=4, max_rallies=20, seed=42,
    title='REINFORCE + baseline: stochastic live play')

### 🏟️ Host a Pong tournament server

You have an agent. So does everyone else in the room. Build a server they can register their model with and let the models play each other on a leaderboard. The exercise is the design.

**Minimal contract.** Each team exposes one HTTP endpoint. The tournament server POSTs the current state, the team responds with an action, within a deadline:

```json
// server -> team
{"state": {"bx": 3, "by": 2, "vy": 1, "py": 2}}

// team -> server  (within 200 ms; 0 = up, 1 = stay, 2 = down)
{"action": 1}
```

A team server is essentially this:

```python
from fastapi import FastAPI
import numpy as np
from viz_helpers import state_index

Q = np.load("my_q_table.npy")
app = FastAPI()

@app.post("/step")
def step(p: dict):
    s = p["state"]
    s_idx = state_index((s["bx"], s["by"], s["vy"], s["py"]))
    return {"action": int(np.argmax(Q[s_idx]))}
```

Anything fancier (a torch policy, a bandit, a hand-coded oracle) just changes the body of `step`.

**The decisions you still owe yourself.** Tournament format (round-robin or Elo ladder), scoring rule (win-rate or aggregate catches), what to do on a timeout or invalid action, how to log every step so the leaderboard is reproducible, and how aggressively to cap inference time so a giant network does not auto-win. None of this is RL; all of it is the gap between a notebook agent and a deployed one.